# Replicação Entre Regiões para Amazon Bedrock AgentCore Memory

## Visão Geral

Este tutorial demonstra como construir **replicação ativa-passiva entre regiões** para o Amazon Bedrock AgentCore Memory usando o recurso de [streaming de registros de memória](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory-record-streaming.html). Quando a região primária transmite um evento de registro de memória, um consumidor Lambda o replica para a região secundária em tempo quase real.

### Detalhes do Tutorial

| Informação | Detalhes |
|:-----------|:--------|
| Tipo do tutorial | Replicação Entre Regiões |
| Recurso | Streaming de Registros de Memória + Consumidor Lambda |
| Recursos-chave | Kinesis, Lambda ESM, CloudFormation, DynamoDB Global Table |
| Complexidade do exemplo | Avançado |
| SDK utilizado | boto3, AWS CLI |

### O Que Você Vai Aprender

1. Implantar infraestrutura de replicação em duas regiões usando CloudFormation
2. Criar instâncias do AgentCore Memory com streaming habilitado
3. Verificar se os registros são replicados da região primária para a secundária em tempo quase real
4. Realizar um failover alternando o streaming entre regiões
5. Limpar todos os recursos

### Arquitetura

```mermaid
flowchart LR
    subgraph primary["Primária (us-east-1)"]
        PM[AgentCore Memory<br/>streaming: ON]
        PK[Kinesis Stream]
        PL[Consumidor Lambda]
        PDLQ[SQS DLQ]
    end

    subgraph secondary["Secundária (us-west-2)"]
        SM[AgentCore Memory<br/>streaming: OFF]
        SK[Kinesis Stream]
        SL["Lambda (inativo)"]
    end

    DDB[(DynamoDB Global Table<br/>ACTIVE_REGION)]

    PM -->|transmite eventos| PK
    PK -->|gatilho ESM| PL
    PL -->|BatchCreateMemoryRecords| SM
    PL -.->|em caso de falha| PDLQ
    SK -.->|sem fluxo de dados| SL

    style primary fill:#e8f5e9,stroke:#2e7d32
    style secondary fill:#fff3e0,stroke:#ef6c00
    style PM fill:#c8e6c9
    style SM fill:#ffe0b2
```

### Fluxo de Replicação

```mermaid
sequenceDiagram
    participant App as Aplicação
    participant PM as Memória Primária
    participant KS as Kinesis Stream
    participant LC as Consumidor Lambda
    participant SM as Memória Secundária

    App->>PM: BatchCreateMemoryRecords
    PM->>KS: Evento MemoryRecordCreated
    KS->>LC: ESM aciona Lambda
    LC->>LC: Verifica prevenção de loop (prefixo replicated/)
    LC->>SM: BatchCreateMemoryRecords (ns: replicated/...)
    Note over SM: Registro armazenado com<br/>namespace replicated/
```

### Fluxo de Failover

```mermaid
sequenceDiagram
    participant Op as Operador
    participant SM as Memória Secundária
    participant PM as Memória Primária
    participant DDB as DynamoDB

    Op->>SM: update-memory (habilita streaming)
    Op->>PM: update-memory (desabilita streaming)
    Op->>DDB: Define ACTIVE_REGION = us-west-2
    Note over SM,PM: Secundária agora é ativa,<br/>replicando para a primária
```

**Decisões de design importantes:**
- Streaming está **DESLIGADO** na secundária — custo zero com eventos de loopback
- Lambda ESM permanece habilitado em ambas as regiões — quando o streaming está desligado, fica inativo
- Failover = alternar streaming (duas chamadas de API, leva segundos)
- Prevenção de loop via prefixo de namespace `replicated/`

### Pré-requisitos

- **Python 3.10+**
- **AWS CLI v2.34+** — As APIs do AgentCore Memory requerem uma versão recente da CLI. Atualize com `brew upgrade awscli` (macOS) ou `pip install --upgrade awscli`
- **boto3 >= 1.42.63** — instalado automaticamente pela primeira célula
- **Amazon Bedrock AgentCore** com acesso habilitado em `us-east-1` e `us-west-2`

### Permissões IAM Necessárias

Suas credenciais AWS precisam de permissões nos seguintes serviços.

| Serviço | Ações Principais | Motivo |
|:--------|:-----------|:----|
| **Bedrock AgentCore** | `CreateMemory`, `DeleteMemory`, `GetMemory`, `ListMemories`, `UpdateMemory`, `BatchCreateMemoryRecords`, `ListMemoryRecords` | Criar/gerenciar instâncias de Memory, ler/escrever registros, configurar streaming |
| **CloudFormation** | `CreateStack`, `UpdateStack`, `DeleteStack`, `DescribeStacks`, `CreateChangeSet`, `ExecuteChangeSet` | Implantar e remover stacks de infraestrutura |
| **Kinesis** | `CreateStream`, `DeleteStream`, `DescribeStream`, `GetShardIterator`, `GetRecords` | Criar streams, ler eventos (Passo 4) |
| **Lambda** | `CreateFunction`, `UpdateFunctionCode`, `UpdateFunctionConfiguration`, `DeleteFunction`, `CreateEventSourceMapping`, `DeleteEventSourceMapping` | Implantar o consumidor de replicação e ESM |
| **IAM** | `CreateRole`, `PutRolePolicy`, `PassRole`, `DeleteRole`, `DeleteRolePolicy` | Criar roles de execução para Lambda e streaming de Memory |
| **SQS** | `CreateQueue`, `DeleteQueue`, `SendMessage`, `GetQueueAttributes` | Criar/gerenciar a Dead Letter Queue |
| **DynamoDB** | `CreateTable`, `DeleteTable`, `PutItem`, `GetItem`, `DescribeTable` | Criar Global Table, ler/escrever configuração da região ativa |
| **S3** | `CreateBucket`, `PutObject`, `GetObject`, `DeleteObject`, `DeleteBucket`, `ListBucket` | Armazenar pacotes de implantação Lambda |
| **STS** | `GetCallerIdentity` | Detectar ID da conta |
| **CloudWatch** | `PutMetricAlarm`, `DeleteAlarms`, `PutDashboard`, `DeleteDashboards` | Criar alarmes de monitoramento e dashboard |

## Passo 0: Configuração do Ambiente

Começamos instalando o SDK necessário e configurando nossas duas regiões-alvo. O notebook usa `us-east-1` como primária e `us-west-2` como secundária por padrão — você pode sobrescrevê-las com as variáveis de ambiente `PRIMARY_REGION` e `SECONDARY_REGION`.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import os, json, time, base64, logging
import boto3
from datetime import datetime, timezone

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Configure regions — override with environment variables if needed
PRIMARY_REGION = os.getenv('PRIMARY_REGION', 'us-east-1')
SECONDARY_REGION = os.getenv('SECONDARY_REGION', 'us-west-2')
ACCOUNT_ID = boto3.client('sts').get_caller_identity()['Account']

# Used by the cleanup cells to find CloudFormation stacks and S3 buckets
STACK_PREFIX = 'agentcore-replication'

print(f'Account: {ACCOUNT_ID}')
print(f'Primary: {PRIMARY_REGION}  Secondary: {SECONDARY_REGION}')

## Passo 1: Implantar Infraestrutura

O script `scripts/deploy.sh` orquestra a implantação completa em ambas as regiões. Veja o que ele faz:

1. **Empacotar Lambda** — Empacota `scripts/handler.py` com suas dependências em um zip e faz upload para o S3 em ambas as regiões
2. **Implantar DynamoDB Global Table** — Uma tabela de registro único (`scripts/global-stack.yaml`) replicada em ambas as regiões, usada para rastrear qual região está atualmente ativa
3. **Implantar stacks por região** — Cada região recebe um Kinesis Data Stream, consumidor Lambda, SQS Dead Letter Queue, roles IAM e alarmes CloudWatch (`scripts/regional-stack.yaml`)
4. **Criar instâncias do AgentCore Memory** — Primária com streaming LIGADO (eventos fluem para o Kinesis), secundária sem streaming (infraestrutura pronta mas inativa)
5. **Conectar IDs entre regiões** — Atualiza as variáveis de ambiente Lambda de cada região com o Memory ID remoto para que saiba onde replicar
6. **Configuração inicial** — Escreve `ACTIVE_REGION = us-east-1` no DynamoDB

Isso leva ~5 minutos. Você verá a saída de progresso de cada etapa abaixo.

In [ ]:
!bash scripts/deploy.sh {PRIMARY_REGION} {SECONDARY_REGION}

## Passo 2: Verificar Implantação

O script `deploy.sh` armazena ambos os IDs de memória na tabela de configuração do DynamoDB. Aqui nós os lemos de volta e confirmamos que ambas as memórias estão `ACTIVE`.

Os IDs de memória capturados aqui (`primary_memory_id` e `secondary_memory_id`) são usados ao longo do restante do notebook.


In [ ]:
def get_memory_id(key):
    """Look up a memory ID from the DynamoDB config table."""
    ddb = boto3.client('dynamodb', region_name=PRIMARY_REGION)
    item = ddb.get_item(
        TableName='AgentCoreMemoryReplicationConfig',
        Key={'PK': {'S': key}}
    ).get('Item', {})
    return item.get('memory_id', {}).get('S')

# The deploy script stores memory IDs in the config table
print('Reading memory IDs from config table...')
primary_memory_id = get_memory_id('MEMORY_ID_PRIMARY')
secondary_memory_id = get_memory_id('MEMORY_ID_SECONDARY')

# Verify both memories are ACTIVE
for label, mid, region in [('Primary', primary_memory_id, PRIMARY_REGION),
                            ('Secondary', secondary_memory_id, SECONDARY_REGION)]:
    if mid:
        client = boto3.client('bedrock-agentcore-control', region_name=region)
        status = client.get_memory(memoryId=mid)['memory']['status']
        print(f'{label} Memory: {mid} ({status})')
    else:
        print(f'{label} Memory: NOT FOUND')

assert primary_memory_id and secondary_memory_id, 'Deployment incomplete — run scripts/deploy.sh first'


In [ ]:
# Verify active region tracking
ddb = boto3.client('dynamodb', region_name=PRIMARY_REGION)
item = ddb.get_item(
    TableName='AgentCoreMemoryReplicationConfig',
    Key={'PK': {'S': 'ACTIVE_REGION'}}
).get('Item', {})

print(f"Active region: {item.get('region', {}).get('S', 'NOT SET')}")

## Passo 3: Testar Replicação

Este é o teste principal. Vamos criar registros de memória na região **primária** e verificar se eles aparecem automaticamente na região **secundária**.

O pipeline de replicação funciona assim:
1. Chamamos `BatchCreateMemoryRecords` na Memory primária
2. Como o streaming está LIGADO, cada registro dispara um evento `MemoryRecordCreated` no Kinesis stream da primária
3. O consumidor Lambda (conectado via Event Source Mapping) captura o evento
4. O Lambda chama `BatchCreateMemoryRecords` na Memory secundária, prefixando o namespace com `replicated/` para prevenir loops
5. O registro aparece na secundária sob o namespace `replicated/`

### 3a. Criar registros de teste na primária

Vamos criar 3 registros com diferentes namespaces para simular memória real de agentes — preferências de usuário e avaliações.

In [ ]:
# Create a client for the primary region's AgentCore Memory data plane
primary_client = boto3.client('bedrock-agentcore', region_name=PRIMARY_REGION)

# Test records simulating real agent memory — preferences and evaluations
test_records = [
    {'text': 'User prefers Python for backend services', 'ns': 'user/alice'},
    {'text': 'User likes event-driven architectures with Lambda', 'ns': 'user/alice'},
    {'text': 'User is evaluating multi-region disaster recovery', 'ns': 'user/bob'},
]

created_ids = []
for i, rec in enumerate(test_records):
    resp = primary_client.batch_create_memory_records(
        memoryId=primary_memory_id,
        records=[{
            'requestIdentifier': f'test-{i}-{int(time.time())}',  # unique ID for idempotency
            'content': {'text': rec['text']},
            'namespaces': [rec['ns']],                            # e.g. user/alice
            'timestamp': str(int(time.time())),                   # epoch seconds as string
        }]
    )
    rid = resp['successfulRecords'][0]['memoryRecordId']
    created_ids.append(rid)
    print(f'Created: {rid} — {rec["text"][:60]}')

print(f'\n✅ Created {len(created_ids)} records in {PRIMARY_REGION}')

### 3b. Aguardar replicação e verificar na secundária

O pipeline de replicação (Memory → Kinesis → Lambda → Memory remota) normalmente leva de 10 a 30 segundos de ponta a ponta. Fazemos polling no namespace `replicated/` da secundária até que todos os 3 registros apareçam.

Se a replicação estiver funcionando, você verá o mesmo texto de registro da primária, agora armazenado na secundária com namespaces prefixados com `replicated/` (ex.: `replicated/user/alice`).

In [ ]:
# Create a client for the secondary region's AgentCore Memory data plane
secondary_client = boto3.client('bedrock-agentcore', region_name=SECONDARY_REGION)

# Poll the secondary for records in the 'replicated/' namespace
# The Lambda consumer prefixes namespaces with 'replicated/' when writing to the remote region
# So 'user/alice' in primary becomes 'replicated/user/alice' in secondary
print('Waiting for replication (polling every 10s, up to 120s)...\n')
start = time.time()
replicated = []

while time.time() - start < 120:
    try:
        resp = secondary_client.list_memory_records(
            memoryId=secondary_memory_id,
            namespace='replicated/'  # prefix match — finds replicated/user/alice, replicated/user/bob, etc.
        )
        replicated = resp.get('memoryRecordSummaries', [])
        if len(replicated) >= len(test_records):
            break
    except Exception:
        pass
    time.sleep(10)

elapsed = time.time() - start
print(f'Found {len(replicated)} replicated record(s) in {elapsed:.0f}s:\n')
for r in replicated:
    text = r.get('content', {}).get('text', 'N/A')[:80]
    print(f"  {r['memoryRecordId']} — {text}")

if len(replicated) >= len(test_records):
    print(f'\n✅ All {len(test_records)} records replicated successfully!')
else:
    print(f'\n⚠️  Only {len(replicated)}/{len(test_records)} replicated. Check Lambda logs for errors.')

## Passo 4: Ler Eventos do Kinesis Stream

Para ter visibilidade do que está acontecendo nos bastidores, vamos ler diretamente do Kinesis stream da primária. Esses são os eventos brutos que o AgentCore Memory publica quando registros mudam — os mesmos eventos que o consumidor Lambda processa.

Você deve ver:
- Um evento `StreamingEnabled` (publicado quando o streaming foi configurado pela primeira vez)
- Eventos `MemoryRecordCreated` para cada registro que criamos no Passo 3

Cada evento inclui o tipo do evento, ID do registro de memória, namespaces e (com nível `FULL_CONTENT`) o texto real do registro.

In [ ]:
# Read raw events from the primary's Kinesis stream
# These are the same events the Lambda consumer processes
kinesis = boto3.client('kinesis', region_name=PRIMARY_REGION)
stream_info = kinesis.describe_stream(StreamName='agentcore-memory-stream')
shard_id = stream_info['StreamDescription']['Shards'][0]['ShardId']

# Start from the oldest available record (TRIM_HORIZON)
iterator = kinesis.get_shard_iterator(
    StreamName='agentcore-memory-stream',
    ShardId=shard_id,
    ShardIteratorType='TRIM_HORIZON'
)['ShardIterator']

events = []
for _ in range(5):  # poll up to 5 times
    resp = kinesis.get_records(ShardIterator=iterator, Limit=100)
    for rec in resp['Records']:
        data = base64.b64decode(rec['Data']) if isinstance(rec['Data'], str) else rec['Data']
        events.append(json.loads(data))
    iterator = resp['NextShardIterator']
    if not resp['Records']:
        time.sleep(2)

print(f'Read {len(events)} event(s) from Kinesis:\n')
for i, evt in enumerate(events[:10]):
    se = evt.get('memoryStreamEvent', {})
    print(f"  [{se.get('eventType','?')}] {se.get('memoryRecordId','N/A')[:40]}  ns={se.get('namespaces',[])}")

## Passo 5: Testar Failover

Agora simulamos uma falha regional alternando a região ativa da primária para a secundária. O processo de failover é:

1. **Habilitar streaming na secundária** — o Kinesis stream da secundária começa a receber eventos, e seu Lambda começa a replicar para a primária
2. **Desabilitar streaming na primária** — a primária para de publicar eventos
3. **Atualizar DynamoDB** — definir `ACTIVE_REGION` para a secundária para que sua camada de aplicação saiba para onde apontar

A ordem importa: habilite o novo caminho **antes** de desabilitar o antigo. Isso garante zero lacuna na replicação. Se ambas as regiões brevemente tiverem streaming ligado, o prefixo de namespace `replicated/` previne loops infinitos.

### 5a. Alternar streaming

In [ ]:
# Enable secondary FIRST — this starts the reverse replication path
# before we cut off the forward path. No replication gap.
!bash scripts/toggle-streaming.sh enable {SECONDARY_REGION}

# Then disable primary — it stops publishing events to Kinesis
!bash scripts/toggle-streaming.sh disable {PRIMARY_REGION}

In [ ]:
# Update active region in DynamoDB
ddb.put_item(
    TableName='AgentCoreMemoryReplicationConfig',
    Item={
        'PK': {'S': 'ACTIVE_REGION'},
        'region': {'S': SECONDARY_REGION},
        'updated_at': {'S': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')},
        'updated_by': {'S': 'notebook-failover'},
    }
)
print(f'Active region updated to: {SECONDARY_REGION}')

### 5b. Verificar failover — escrever na secundária, verificar replicação na primária

Com a secundária agora ativa, vamos provar que a replicação funciona na direção reversa. Vamos criar um registro na secundária e verificar se ele aparece no namespace `replicated/` da primária.

Isso confirma o failover completo: a secundária agora é a fonte da verdade, e a primária está recebendo dados replicados.

In [ ]:
# Write a record to the secondary (now the active region)
resp = secondary_client.batch_create_memory_records(
    memoryId=secondary_memory_id,
    records=[{
        'requestIdentifier': f'failover-test-{int(time.time())}',
        'content': {'text': 'Record created during failover in secondary region'},
        'namespaces': ['user/failover-test'],
        'timestamp': str(int(time.time())),
    }]
)
failover_id = resp['successfulRecords'][0]['memoryRecordId']
print(f'Created in secondary: {failover_id}')

# Poll the primary for the replicated record
# The secondary's Lambda is now replicating to the primary
print('Waiting for replication to primary (polling every 10s, up to 120s)...\n')
start = time.time()
recs = []
while time.time() - start < 120:
    try:
        recs = primary_client.list_memory_records(
            memoryId=primary_memory_id,
            namespace='replicated/'
        ).get('memoryRecordSummaries', [])
        if recs:
            break
    except Exception:
        pass
    time.sleep(10)

if recs:
    for r in recs:
        print(f"  {r['memoryRecordId']} — {r.get('content',{}).get('text','')[:80]}")
    print(f'\n✅ Failover replication working! ({time.time()-start:.0f}s)')
else:
    print('⚠️  No replicated records yet — check Lambda logs in secondary region')

### 5c. Failback — restaurar primária como ativa

O failback é o mesmo processo ao contrário: habilitar streaming na primária, desabilitar na secundária, atualizar DynamoDB. Após isso, o sistema volta à sua configuração original.

In [ ]:
# Failback: restore original configuration
# Same process in reverse — enable primary, disable secondary
!bash scripts/toggle-streaming.sh enable {PRIMARY_REGION}
!bash scripts/toggle-streaming.sh disable {SECONDARY_REGION}

# Update the config table so the application layer knows primary is active again
ddb.put_item(
    TableName='AgentCoreMemoryReplicationConfig',
    Item={
        'PK': {'S': 'ACTIVE_REGION'},
        'region': {'S': PRIMARY_REGION},
        'updated_at': {'S': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')},
        'updated_by': {'S': 'notebook-failback'},
    }
)
print(f'\n✅ Failback complete. Active region: {PRIMARY_REGION}')

## Passo 6: Limpeza

Exclua todos os recursos criados por este tutorial. Isso remove:
- Instâncias do AgentCore Memory em ambas as regiões (aguarda a conclusão da exclusão)
- Stacks do CloudFormation (Kinesis, Lambda, SQS, IAM, CloudWatch) em ambas as regiões
- DynamoDB Global Table
- Buckets S3 usados para pacotes de implantação Lambda

> **Nota sobre custos:** Kinesis Data Streams geram cobranças por hora por shard (~$11/mês cada). Certifique-se de executar a limpeza quando terminar para evitar custos contínuos.

In [ ]:
# Delete AgentCore Memory instances
for region, mid in [(PRIMARY_REGION, primary_memory_id), (SECONDARY_REGION, secondary_memory_id)]:
    try:
        client = boto3.client('bedrock-agentcore-control', region_name=region)
        client.delete_memory(memoryId=mid)
        print(f'Deleting memory {mid} in {region}...')
        # Wait for deletion
        for _ in range(30):
            try:
                status = client.get_memory(memoryId=mid)['memory']['status']
                if status == 'DELETING':
                    time.sleep(5)
                else:
                    break
            except client.exceptions.ResourceNotFoundException:
                print(f'  ✅ Deleted')
                break
    except Exception as e:
        print(f'  Error: {e}')

In [ ]:
# Delete CloudFormation stacks
for region in [PRIMARY_REGION, SECONDARY_REGION]:
    cf = boto3.client('cloudformation', region_name=region)
    try:
        cf.delete_stack(StackName=f'{STACK_PREFIX}-regional')
        print(f'Deleting regional stack in {region}...')
    except Exception as e:
        print(f'  Error: {e}')

# Global stack (only in primary)
try:
    cf_primary = boto3.client('cloudformation', region_name=PRIMARY_REGION)
    cf_primary.delete_stack(StackName=f'{STACK_PREFIX}-global')
    print(f'Deleting global stack...')
except Exception as e:
    print(f'  Error: {e}')

# Clean up S3 buckets
for region in [PRIMARY_REGION, SECONDARY_REGION]:
    bucket = f'{STACK_PREFIX}-artifacts-{ACCOUNT_ID}-{region}'
    try:
        s3 = boto3.client('s3', region_name=region)
        objs = s3.list_objects_v2(Bucket=bucket).get('Contents', [])
        for obj in objs:
            s3.delete_object(Bucket=bucket, Key=obj['Key'])
        s3.delete_bucket(Bucket=bucket)
        print(f'Deleted S3 bucket: {bucket}')
    except Exception as e:
        print(f'  S3 cleanup ({bucket}): {e}')

print('\n✅ Cleanup complete')

## Conclusão

Neste tutorial você construiu replicação entre regiões de ponta a ponta para o AgentCore Memory:

1. **Implantou infraestrutura** — Kinesis streams, consumidores Lambda, SQS DLQs, roles IAM e alarmes CloudWatch em duas regiões
2. **Criou instâncias de Memory** — primária com streaming LIGADO, secundária com streaming DESLIGADO
3. **Verificou a replicação** — registros criados na primária apareceram no namespace `replicated/` da secundária
4. **Testou failover/failback** — alternou o streaming entre regiões em segundos

### Principais Conclusões

| Métrica | Valor |
|:-------|:------|
| RPO (Recovery Point Objective) | 5–15 segundos |
| RTO (Recovery Time Objective) | 15–30 segundos |
| Mecanismo de failover | Alternar streaming via API `update-memory` |
| Prevenção de loop | Prefixo de namespace `replicated/` |
| Resolução de conflitos | Consolidação nativa do AgentCore Memory |

### Próximos Passos

- Adicionar alarmes CloudWatch para detecção automatizada de failover
- Integrar com health checks do Route 53 para failover a nível de DNS
- Estender para topologias de 3+ regiões
- Adicionar suporte a replicação entre contas
